In [19]:
# data profiling
import pandas as pd
import os


In [20]:

def load_csv(file_name):
    data_path = 'C:/Users/DELL/Projects/ecommerce-sales-pipeline/data/raw'
    raw=os.path.join(data_path,file_name)
    return pd.read_csv(raw,encoding='latin1', sep=";")

Customers= load_csv('Customers.csv')
Location= load_csv('Location.csv')
Products = load_csv('Products.csv')
Orders= load_csv('Orders.csv')


#reusable csv loader to avoid repeating file-loading logic across datasets
#encoding is set to latin1 because the csv is not encoded in UTF-8, hence errors when reading the csv file 
# use sep=';' so that  we can split fields. But csv are comma-delimited by default

In [21]:
def profile_col(df):
    profile = {
        "Data types": df.dtypes,
        "Missing values":df.isna().sum(),
        "Missing percentage": (df.isna().mean() * 100 ).round(2),
        "Unique values": df.nunique(),
        "duplicate":df.duplicated().sum()
    }
    return pd.DataFrame(profile)

profile_col(Products)


,Data types,Missing values,Missing percentage,Unique values,duplicate
Product ID,object,0,0.00,1869,0
Category,object,148,7.81,3,0
Sub-Category,object,148,7.81,17,0
"Product Name,,,,,",object,148,7.81,1702,0


In [22]:


def uncover_location(df2):
    return pd.DataFrame({
        "Metric": [
            "Rows with missing values",
            "Duplicate postal codes",
        ],
        "Count": [
            df2.isna().any(axis=1).sum(),
            df2.duplicated().sum(),
        ],
    })

uncover_location(Location)

,Metric,Count
0,Rows with missing values,1
1,Duplicate postal codes,0


In [23]:
Products[Products.isna().any(axis=1)]

,Product ID,Category,Sub-Category,"Product Name,,,,,"
26,"FUR-BO-10002916;Furniture;Bookcases;""Rush Hier...",NaN,NaN,NaN
142,"FUR-FU-10000087;Furniture;Furnishings;""Executi...",NaN,NaN,NaN
147,"FUR-FU-10000222;Furniture;Furnishings;""Seth Th...",NaN,NaN,NaN
149,"FUR-FU-10000260;Furniture;Furnishings;""6"""" Cub...",NaN,NaN,NaN
152,"FUR-FU-10000305;Furniture;Furnishings;""Tenex V...",NaN,NaN,NaN
...,...,...,...,...
1479,"OFF-SU-10004782;Office Supplies;Supplies;""Elit...",NaN,NaN,NaN
1615,"TEC-AC-10004659;Technology;Accessories;""Imatio...",NaN,NaN,NaN
1657,"TEC-MA-10001127;Technology;Machines;""HP Design...",NaN,NaN,NaN
1731,"TEC-PH-10000702;Technology;Phones;""Square Cred...",NaN,NaN,NaN


In [24]:
#referential integrity check for customer id in orders 

invalid_orders= Orders[~Orders['Customer ID'].isin(Customers['Customer ID'])]

print(len(invalid_orders))

0


In [25]:
#referential integrity check for product id in orders 

invalid_products= Orders[~Orders['Product ID'].isin(Products['Product ID'])]

print(invalid_products)

      Row ID        Order ID  Order Date   Ship Date       Ship Mode  \
16        17  CA-2020-105893  11/11/2020  18/11/2020  Standard Class   
20        21  CA-2020-143336  27/08/2020  01/09/2020    Second Class   
32        33  US-2021-150630  17/09/2021  21/09/2021  Standard Class   
37        38  CA-2021-117415  27/12/2021  31/12/2021  Standard Class   
63        64  CA-2021-135545  24/11/2021  30/11/2021  Standard Class   
...      ...             ...         ...         ...             ...   
9928    9929  CA-2022-129630  04/09/2022  04/09/2022        Same Day   
9956    9957  US-2020-143287  11/11/2020  17/11/2020  Standard Class   
9963    9964  CA-2021-143700  26/07/2021  26/07/2021        Same Day   
9985    9986  CA-2021-100251  17/05/2021  23/05/2021  Standard Class   
9992    9993  CA-2023-121258  26/02/2023  03/03/2023  Standard Class   

     Customer ID      Segment  Postal Code       Product ID    Sales  \
16      PK-19075     Consumer        53711  OFF-ST-10004186  66

In [26]:
#referential integrity check for postal code in orders 

invalid_postal= Orders[~Orders['Postal Code'].isin(Location['Postal Code'])]

print(len(invalid_postal))

0


In [27]:
is_unique = not Orders.duplicated(subset=['Order ID','Product ID']).any()
print(is_unique)

False


In [28]:
def latest_date(df):
    recent_date = df.max()
    return recent_date

latest_date(Orders['Order Date'])
latest_date(Orders['Ship Date'])


'31/12/2023'

In [29]:

def oldest_date(df):
    oldest = df.min()
    return oldest

oldest_date(Orders['Ship Date'])
oldest_date(Orders['Order Date'])



'01/01/2023'

In [30]:
(Orders['Ship Date'] < Orders['Order Date']).any()

np.True_

In [31]:
Orders[Orders.duplicated(subset=['Order ID','Product ID'], keep=False)].head(10)


#Why keep=False? By default, duplicated() hides the first instance of a repeated pair. Using keep=False ensures both the original row and its
#duplicates are included in the results so you can inspect them together.

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Segment,Postal Code,Product ID,Sales,Quantity,Discount,Profit
350,351,CA-2022-129714,01/09/2022,03/09/2022,First Class,AB-10060,Home Office,10009,OFF-PA-10001970,24.560,2,0.0,11.5432
352,353,CA-2022-129714,01/09/2022,03/09/2022,First Class,AB-10060,Home Office,10009,OFF-PA-10001970,49.120,4,0.0,23.0864
430,431,US-2022-123750,15/04/2022,21/04/2022,Standard Class,RB-19795,Home Office,28052,TEC-AC-10004659,408.744,7,0.2,76.6395
431,432,US-2022-123750,15/04/2022,21/04/2022,Standard Class,RB-19795,Home Office,28052,TEC-AC-10004659,291.960,5,0.2,54.7425
1300,1301,CA-2022-137043,23/12/2022,25/12/2022,Second Class,LC-17140,Consumer,22153,FUR-FU-10003664,572.760,6,0.0,166.1004
1301,1302,CA-2022-137043,23/12/2022,25/12/2022,Second Class,LC-17140,Consumer,22153,FUR-FU-10003664,286.380,3,0.0,83.0502
3183,3184,CA-2023-152912,09/11/2023,12/11/2023,Second Class,BM-11650,Corporate,21044,OFF-ST-10003208,1633.140,9,0.0,473.6106
3184,3185,CA-2023-152912,09/11/2023,12/11/2023,Second Class,BM-11650,Corporate,21044,OFF-ST-10003208,544.380,3,0.0,157.8702
3405,3406,US-2020-150119,23/04/2020,27/04/2020,Standard Class,LB-16795,Home Office,43229,FUR-CH-10002965,281.372,2,0.3,-12.0588
3406,3407,US-2020-150119,23/04/2020,27/04/2020,Standard Class,LB-16795,Home Office,43229,FUR-CH-10002965,281.372,2,0.3,-12.0588


In [32]:
pro_len=Products[Products.duplicated(subset=['Product ID'])].head(25)
print(pro_len)

           Product ID         Category Sub-Category  \
19    FUR-BO-10002213        Furniture    Bookcases   
67    FUR-CH-10001146        Furniture       Chairs   
186   FUR-FU-10001473        Furniture  Furnishings   
396   OFF-AP-10000576  Office Supplies   Appliances   
516   OFF-AR-10001149  Office Supplies          Art   
732   OFF-BI-10002026  Office Supplies      Binders   
843   OFF-BI-10004632  Office Supplies      Binders   
845   OFF-BI-10004654  Office Supplies      Binders   
1058  OFF-PA-10000357  Office Supplies        Paper   
1064  OFF-PA-10000477  Office Supplies        Paper   
1080  OFF-PA-10000659  Office Supplies        Paper   
1103  OFF-PA-10001166  Office Supplies        Paper   
1162  OFF-PA-10001970  Office Supplies        Paper   
1219  OFF-PA-10003022  Office Supplies        Paper   
1353  OFF-ST-10001228  Office Supplies      Storage   
1441  OFF-ST-10004950  Office Supplies      Storage   
1541  TEC-AC-10002049       Technology  Accessories   
1558  TEC-

In [33]:
earliest = Orders['Ship Date'].min()
latest = Orders['Ship Date'].max()
print(f"Earliest date:",{earliest})
print(f"Latest date:",{latest})

Earliest date: {'01/01/2021'}
Latest date: {'31/12/2023'}


In [34]:
earliest = Orders['Ship Date'].min()
latest = Orders['Ship Date'].max()
print(f"Earliest date:",{earliest})
print(f"Latest date:",{latest})

Earliest date: {'01/01/2021'}
Latest date: {'31/12/2023'}


In [36]:
Orders[Orders.duplicated( keep=False)].head(10)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Segment,Postal Code,Product ID,Sales,Quantity,Discount,Profit
